# Parts 4–5 — Justified Preprocessing and Exploratory Data Analysis

This notebook explains every transformation before applying or inspecting it. It then studies article domains, passage lengths, chunking alternatives, and diagnostic-query coverage to identify likely modeling difficulties.

**Notebook status:** executed from frozen local artifacts; no model fitting and no locked-test evaluation.


In [1]:
from collections import Counter
from pathlib import Path
import csv
import hashlib
import html
import json
import platform
import random
import statistics

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SEED = 20250816
random.seed(SEED)
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

def load_json(relative_path):
    return json.loads((ROOT / relative_path).read_text(encoding="utf-8"))

def print_table(rows, columns):
    if not rows:
        print("(no rows)")
        return
    widths = {
        column: max(len(str(column)), *(len(str(row.get(column, ""))) for row in rows))
        for column in columns
    }
    print(" | ".join(str(column).ljust(widths[column]) for column in columns))
    print("-+-".join("-" * widths[column] for column in columns))
    for row in rows:
        print(" | ".join(str(row.get(column, "")).ljust(widths[column]) for column in columns))

def write_bar_svg(filename, values, title, *, maximum=None):
    values = list(values)
    width, left, right, row_height = 820, 245, 80, 34
    height = 76 + row_height * len(values)
    plot_width = width - left - right
    largest = maximum or max((float(value) for _, value in values), default=1.0) or 1.0
    elements = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="white"/>',
        f'<text x="{width / 2}" y="27" text-anchor="middle" font-family="Arial" font-size="18" font-weight="700">{html.escape(title)}</text>',
    ]
    for index, (label, value) in enumerate(values):
        y = 52 + index * row_height
        bar_width = plot_width * float(value) / largest
        elements.extend([
            f'<text x="{left - 10}" y="{y + 17}" text-anchor="end" font-family="Arial" font-size="13">{html.escape(str(label))}</text>',
            f'<rect x="{left}" y="{y}" width="{bar_width:.2f}" height="20" rx="3" fill="#1c5b58"/>',
            f'<text x="{min(left + bar_width + 7, width - 58):.2f}" y="{y + 16}" font-family="Arial" font-size="12">{float(value):.4g}</text>',
        ])
    elements.append('</svg>')
    target = FIGURES / filename
    target.write_text("\n".join(elements) + "\n", encoding="utf-8")
    print(f"Saved visualization: {target.relative_to(ROOT)}")
    return target

print(f"Project: {ROOT.name} | Python: {platform.python_version()} | fixed seed: {SEED}")


Project: h | Python: 3.12.13 | fixed seed: 20250816


## Part 4 — Preprocessing pipeline and justification

1. **Preserve raw text and provenance.** Article ID, title, URL, source text, and deterministic checksums remain available so every answer can be traced to its origin.
2. **Unicode NFKC normalization.** Urdu text can contain compatibility variants. NFKC reduces accidental code-point mismatch while the raw form remains stored for audit.
3. **Remove only control characters.** Retrieval should not be distorted by invisible controls; Urdu letters and punctuation must remain intact.
4. **Exclude articles shorter than 80 whitespace tokens.** Very short pages are frequently navigation or fragmentary content and offer little evidence. This threshold is frozen in configuration.
5. **Chunk into 150-token passages with 30-token overlap.** Dense encoders have finite context, while overlap reduces the chance that an answer-spanning sentence is split at a boundary. Development alternatives of 120/24 and 180/36 are retained for comparison.
6. **Attach passage metadata.** Each chunk stores its article, title, URL, span, domain, and deterministic passage ID; this is required for evaluation and citations.
7. **Do not use gold fields during preprocessing or query reformulation.** Gold passage IDs and evidence exist only for scoring. The locked test partition remains untouched until final configuration freeze.


In [2]:
articles = [json.loads(line) for line in (ROOT / "data/raw/wikipedia.jsonl").open(encoding="utf-8") if line.strip()]
passages = [json.loads(line) for line in (ROOT / "data/processed/passages_150_30.jsonl").open(encoding="utf-8") if line.strip()]
print({"articles": len(articles), "passages": len(passages), "represented_articles": len({row["article_id"] for row in passages}), "missing_provenance": sum(not all(row.get(field) for field in ("passage_id", "article_id", "title", "url")) for row in passages)})
assert len(articles) == 4000 and len(passages) == 16352
assert len({row["article_id"] for row in passages}) == 4000


{'articles': 4000, 'passages': 16352, 'represented_articles': 4000, 'missing_provenance': 0}


## Part 5 — Corpus coverage by domain

Domain balance matters because a model can appear effective if the evaluation overrepresents easier topics. Geography and general articles are the largest groups, while science is the smallest. Results should therefore be reported overall and by query category rather than interpreted as uniform Urdu coverage.


In [3]:
article_domains = Counter(row["domain"] for row in articles)
passage_domains = Counter(row["domain"] for row in passages)
rows = [{"domain": domain, "articles": article_domains[domain], "passages": passage_domains[domain], "passages/article": round(passage_domains[domain] / article_domains[domain], 2)} for domain in sorted(article_domains)]
print_table(rows, ["domain", "articles", "passages", "passages/article"])
write_bar_svg("eda_article_domains.svg", sorted(article_domains.items()), "Urdu Wikipedia articles by domain")


domain    | articles | passages | passages/article
----------+----------+----------+-----------------
culture   | 460      | 1921     | 4.18            
general   | 900      | 2265     | 2.52            
geography | 938      | 2943     | 3.14            
history   | 757      | 5644     | 7.46            
pakistan  | 748      | 2500     | 3.34            
science   | 197      | 1079     | 5.48            
Saved visualization: reports\figures\eda_article_domains.svg


![Article counts by domain](../reports/figures/eda_article_domains.svg)


## Passage-length distribution

Passage length controls the context/noise trade-off. Short chunks may omit needed context; long chunks contain more irrelevant sentences and cost more to encode. The summary below uses the actual default passage file.


In [4]:
token_counts = [row["token_count"] for row in passages]
bins = [("31–60", 31, 60), ("61–90", 61, 90), ("91–120", 91, 120), ("121–149", 121, 149), ("150", 150, 150)]
histogram = [(label, sum(low <= value <= high for value in token_counts)) for label, low, high in bins]
print({"minimum": min(token_counts), "median": statistics.median(token_counts), "mean": round(statistics.fmean(token_counts), 3), "maximum": max(token_counts)})
print_table([{"tokens": label, "passages": count} for label, count in histogram], ["tokens", "passages"])
write_bar_svg("eda_passage_lengths.svg", histogram, "Default passage-length distribution")


{'minimum': 31, 'median': 150.0, 'mean': 135.92, 'maximum': 150}
tokens  | passages
--------+---------
31–60   | 735     
61–90   | 1227    
91–120  | 1157    
121–149 | 857     
150     | 12376   
Saved visualization: reports\figures\eda_passage_lengths.svg


![Passage-length distribution](../reports/figures/eda_passage_lengths.svg)


## Chunking sensitivity

All three settings preserve all 4,000 articles. Smaller chunks create more candidates and finer evidence boundaries; larger chunks reduce index size but add irrelevant context. The middle setting is selected as a CPU-friendly compromise, not because the alternatives are discarded.


In [5]:
manifest = load_json("artifacts/metadata/phase1_manifest.json")
chunk_rows = [{"file": Path(item["path"]).name, "passages": item["passages"], "mean_tokens": item["tokens"]["mean"], "median_tokens": item["tokens"]["median"], "articles": item["represented_articles"]} for item in manifest["passage_variants"]]
print_table(chunk_rows, ["file", "passages", "mean_tokens", "median_tokens", "articles"])


file                  | passages | mean_tokens | median_tokens | articles
----------------------+----------+-------------+---------------+---------
passages_120_24.jsonl | 20106    | 111.34      | 120.0         | 4000    
passages_150_30.jsonl | 16352    | 135.92      | 150.0         | 4000    
passages_180_36.jsonl | 13908    | 158.81      | 180.0         | 4000    


## Diagnostic-query composition

The diagnostic set deliberately includes clean, informal, abbreviated, highly noisy, code-switched, named-entity, short, and slightly ambiguous queries. This is a challenge set rather than a population sample; category counts should not be interpreted as real-world prevalence.


In [6]:
with (ROOT / "data/diagnostic/raabta_diagnostic.csv").open(encoding="utf-8-sig", newline="") as handle:
    diagnostic = list(csv.DictReader(handle))
split_counts = Counter(row["split"] for row in diagnostic)
query_types = Counter(row["query_type"] for row in diagnostic)
query_lengths = [len(row["roman_urdu_query"].split()) for row in diagnostic]
print("Splits:", dict(split_counts))
print("Roman-query words:", {"min": min(query_lengths), "median": statistics.median(query_lengths), "mean": round(statistics.fmean(query_lengths), 2), "max": max(query_lengths)})
print_table([{"query_type": key, "questions": value} for key, value in sorted(query_types.items())], ["query_type", "questions"])
write_bar_svg("eda_query_types.svg", sorted(query_types.items()), "Diagnostic questions by query type")


Splits: {'development': 120, 'test': 60}
Roman-query words: {'min': 2, 'median': 5.0, 'mean': 5.01, 'max': 11}
query_type                  | questions
----------------------------+----------
abbreviated_roman_urdu      | 23       
clean_roman_urdu            | 23       
highly_noisy_roman_urdu     | 23       
informal_spelling           | 23       
named_entity                | 22       
short_query                 | 22       
slightly_ambiguous          | 22       
urdu_english_code_switching | 22       
Saved visualization: reports\figures\eda_query_types.svg


![Diagnostic query types](../reports/figures/eda_query_types.svg)


## EDA conclusions and expected modeling difficulties

- Corpus coverage is uneven across domains, so aggregate metrics alone can hide weak categories.
- Most default chunks reach the 150-token ceiling; reranking must separate the relevant sentence from surrounding context.
- Short and noisy Roman queries contain little lexical evidence, and transliteration can amplify spelling errors.
- Code switching requires preserving English tokens while converting Urdu tokens.
- Named entities are especially sensitive to omitted vowels and script conversion, motivating character-level title matching.
- The dataset is small and deliberately constructed, so results are engineering evidence rather than broad population estimates.
